In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd


In [3]:
# Path to original IFND dataset
raw_path = "/content/drive/MyDrive/MP_Project/IFND_dataset/IFND.csv"

# Read dataset safely (Colab compatible)
df_raw = pd.read_csv(
    raw_path,
    encoding="utf-8",
    encoding_errors="replace"
)

# Basic sanity checks
print("Dataset shape:", df_raw.shape)
print("\nColumn names:")
print(df_raw.columns)

print("\nSample rows:")
df_raw.head()


Dataset shape: (56714, 7)

Column names:
Index(['id', 'Statement', 'Image', 'Web', 'Category', 'Date', 'Label'], dtype='object')

Sample rows:


,id,Statement,Image,Web,Category,Date,Label
0,2,"WHO praises India's Aarogya Setu app, says it ...",https://cdn.dnaindia.com/sites/default/files/s...,DNAINDIA,COVID-19,Oct-20,TRUE
1,3,"In Delhi, Deputy US Secretary of State Stephen...",https://cdn.dnaindia.com/sites/default/files/s...,DNAINDIA,VIOLENCE,Oct-20,TRUE
2,4,LAC tensions: China's strategy behind delibera...,https://cdn.dnaindia.com/sites/default/files/s...,DNAINDIA,TERROR,Oct-20,TRUE
3,5,India has signed 250 documents on Space cooper...,https://cdn.dnaindia.com/sites/default/files/s...,DNAINDIA,COVID-19,Oct-20,TRUE
4,6,Tamil Nadu chief minister's mother passes away...,https://cdn.dnaindia.com/sites/default/files/s...,DNAINDIA,ELECTION,Oct-20,TRUE


In [4]:
statement_131 = df_raw.loc[df_raw["id"] == 131, "Statement"].values[0]
print(statement_131)

Both the BJP & Congress submit details of Electoral Bond donors after the Supreme Court�s due date?


In [5]:
df_raw.loc[df_raw["id"] == 132, "Statement"].values[0]

'Amidst the debate around �One Nation, One Election�, here is a look at other Electoral Reform proposals?'

In [6]:
df_raw.loc[df_raw["id"] == 156, "Statement"].values[0]

'What do alphabets �E�, �S�, �Q�, �R�, �M� on the Electoral Roll indicate?'

In [7]:
df_raw.loc[df_raw["id"] == 39785, "Statement"].values[0]

'‘The Truth Behind COVID-19’ Pamphlet is Spreading Bizarre Rumours; Fact Check'

In [8]:
df_raw.loc[df_raw["id"] == 37523, "Statement"].values[0]

'�No cash award system� � Army junks reports that said Shopian encounter staged for Rs 20 lakh'

In [9]:
df_raw.loc[df_raw["id"] == 12989, "Statement"].values[0]

'Kanpur shootout: Cop killer and Vikas Dubey�۪s accomplice held'

In [10]:
import re
import unicodedata

In [11]:
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Fix corrupted possessive cases (very controlled)
    text = re.sub(r"([A-Za-z])�۪s\b", r"\1's", text)
    text = re.sub(r"�s\b", "'s", text)

    # Remove remaining replacement characters
    text = text.replace("�", "")

    # Remove Arabic / non-English combining marks
    text = re.sub(r"[\u0600-\u06FF\u08A0-\u08FF]", "", text)

    # Normalize smart quotes
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    # Normalize dashes (Excel-safe)
    text = text.replace("–", "-").replace("—", "-")

    # Replace ampersand
    text = text.replace("&", " and ")

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [12]:
df_raw["clean_text"] = df_raw["Statement"].apply(clean_text)

In [13]:
for test_id in [131, 132, 156, 12989, 37523, 12987, 47, 255, 229, 174, 179, 185, 123, 48486, 48487, 39785]:
    print("ID:", test_id)
    print(df_raw.loc[df_raw["id"] == test_id, "clean_text"].values[0])
    print("-" * 80)


ID: 131
Both the BJP and Congress submit details of Electoral Bond donors after the Supreme Court's due date?
--------------------------------------------------------------------------------
ID: 132
Amidst the debate around One Nation, One Election, here is a look at other Electoral Reform proposals?
--------------------------------------------------------------------------------
ID: 156
What do alphabets E, S, Q, R, M on the Electoral Roll indicate?
--------------------------------------------------------------------------------
ID: 12989
Kanpur shootout: Cop killer and Vikas Dubey's accomplice held
--------------------------------------------------------------------------------
ID: 37523
No cash award system Army junks reports that said Shopian encounter staged for Rs 20 lakh
--------------------------------------------------------------------------------
ID: 12987
Covid-19 War Room to come up in Delhi to keep eye on city's fight against virus
----------------------------------------

In [14]:
df_raw["Label"] = df_raw["Label"].astype(str).str.upper().str.strip()

df_raw["Label"] = df_raw["Label"].replace({
    "TRUE": "TRUE",
    "FAKE": "FALSE"
})

print(df_raw["Label"].value_counts())


Label
TRUE     37800
FALSE    18914
Name: count, dtype: int64


In [15]:
final_df = df_raw[["id", "clean_text", "Label"]].copy()
print(final_df.shape)
final_df.head()


(56714, 3)


,id,clean_text,Label
0,2,"WHO praises India's Aarogya Setu app, says it ...",TRUE
1,3,"In Delhi, Deputy US Secretary of State Stephen...",TRUE
2,4,LAC tensions: China's strategy behind delibera...,TRUE
3,5,India has signed 250 documents on Space cooper...,TRUE
4,6,Tamil Nadu chief minister's mother passes away...,TRUE


In [16]:
save_path = "/content/drive/MyDrive/MP_Project/IFND_dataset/finally_IFND_is_cleaned.csv"
final_df.to_csv(save_path, index=False)

print("Saved cleaned dataset at:", save_path)


Saved cleaned dataset at: /content/drive/MyDrive/MP_Project/IFND_dataset/finally_IFND_is_cleaned.csv


In [17]:
for test_id in [131, 132, 156, 12989, 37523, 12987, 47, 255, 229, 174, 179, 185, 123, 48486, 48487, 39785]:
    print("ID:", test_id)
    print(df_raw.loc[df_raw["id"] == test_id, "clean_text"].values[0])
    print("-" * 80)

ID: 131
Both the BJP and Congress submit details of Electoral Bond donors after the Supreme Court's due date?
--------------------------------------------------------------------------------
ID: 132
Amidst the debate around One Nation, One Election, here is a look at other Electoral Reform proposals?
--------------------------------------------------------------------------------
ID: 156
What do alphabets E, S, Q, R, M on the Electoral Roll indicate?
--------------------------------------------------------------------------------
ID: 12989
Kanpur shootout: Cop killer and Vikas Dubey's accomplice held
--------------------------------------------------------------------------------
ID: 37523
No cash award system Army junks reports that said Shopian encounter staged for Rs 20 lakh
--------------------------------------------------------------------------------
ID: 12987
Covid-19 War Room to come up in Delhi to keep eye on city's fight against virus
----------------------------------------